# Assignment 1

## Task 1 – Evaluation Rubric

### Criterion Definitions

**Fluency**
- good: The text flows naturally, is easy to read, and has no awkward phrasing.
- ok: The text is mostly readable but contains minor awkward phrasing.
- bad: The text is difficult to read or unnatural.

**Grammar**
- good: No spelling, grammar, or punctuation errors.
- ok: Minor errors that do not affect understanding.
- bad: Frequent or serious errors.

**Tone**
- good: Friendly, persuasive, and professional sales tone.
- ok: Neutral but acceptable tone.
- bad: Inappropriate tone (robotic, too casual, or not persuasive).

**Length**
- good: 50–90 words.
- ok: 40–49 or 91–110 words.
- bad: Outside these ranges.

**Grounding**
- good: Uses only the provided product information.
- ok: Minor interpretation without adding new facts.
- bad: Includes unsupported or hallucinated information.

**Latency**
- good: Less than 2000 ms.
- ok: 2000–5000 ms.
- bad: More than 5000 ms.

**Cost**
- good: Low cost, suitable for scale.
- ok: Moderate cost.
- bad: High cost.

### Pass / Fail Definition

A description passes if:
- At least 4 criteria are rated as "good"
- No criteria are rated as "bad"

### Go / No-Go Rules

- If grounding is not rated as "good" → automatic fail
- If grammar is rated as "bad" → automatic fail

In [ ]:
def final_score(scores):
    if scores["grounding"] != "good":
        return "fail"
    if scores["grammar"] == "bad":
        return "fail"

    good_count = list(scores.values()).count("good")
    bad_count = list(scores.values()).count("bad")

    if good_count >= 4 and bad_count <= 1:
        return "pass"

    return "fail"

## Task 2 – Generate Product Descriptions

In [ ]:
import os
import time

import httpx
import pandas as pd
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam


In [ ]:
from pathlib import Path

dataset_path = Path("Assignment_01_product_dataset.csv")
df = pd.read_csv(str(dataset_path))
df.head()

In [ ]:
SYSTEM_PROMPT = """
You are a professional e-commerce copywriter.

Write a persuasive 50–90 word product description based only on the provided product information.

Requirements:
- Use a friendly and credible sales tone
- Use only the provided information
- Do not invent any features or claims
- Return only the final description text
""".strip()

In [ ]:
api_key = os.getenv("NEBIUS_API_KEY")

if not api_key:
    raise ValueError("NEBIUS_API_KEY is not set. Define it in your environment before running this notebook.")

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=api_key,
    http_client=httpx.Client(verify=False)
)

In [ ]:
def build_user_prompt(product_row):
    return f"""
Product name: {product_row['product_name']}
Structured attributes: {product_row['Product_attribute_list']}
Material: {product_row['material']}
Warranty: {product_row['warranty']}
""".strip()

In [ ]:
def generate_description(product_row, model_name="meta-llama/Meta-Llama-3.1-8B-Instruct-fast"):
    user_prompt = build_user_prompt(product_row)

    start_time = time.time()

    messages: list[ChatCompletionMessageParam] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    response = client.chat.completions.create(
        model=model_name,
        temperature=0.4,
        max_tokens=140,
        messages=messages,
    )

    latency_ms = int((time.time() - start_time) * 1000)

    generated_description = response.choices[0].message.content.strip()

    input_tokens = response.usage.prompt_tokens if response.usage else None
    output_tokens = response.usage.completion_tokens if response.usage else None

    return {
        "model": model_name,
        "generated_description": generated_description,
        "latency_ms": latency_ms,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }

In [ ]:
def generate_description(product_row):
    user_prompt = build_user_prompt(product_row)

    start_time = time.time()

    messages: list[ChatCompletionMessageParam] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    response = client.chat.completions.create(
        model="google/gemma-2-9b-it-fast",
        temperature=0.4,
        max_tokens=140,
        messages=messages,
    )

    latency_ms = int((time.time() - start_time) * 1000)

    description = response.choices[0].message.content.strip()

    input_tokens = None
    output_tokens = None

    if hasattr(response, "usage") and response.usage is not None:
        input_tokens = getattr(response.usage, "prompt_tokens", None)
        output_tokens = getattr(response.usage, "completion_tokens", None)

    return {
        "model": "google/gemma-2-9b-it-fast",
        "generated_description": description,
        "latency_ms": latency_ms,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }

In [ ]:
results = []

for _, product_row in df.iterrows():
    generated = generate_description(product_row)

    output_row = product_row.to_dict()
    output_row.update(generated)

    output_row["fluency"] = ""
    output_row["grammar"] = ""
    output_row["tone"] = ""
    output_row["length"] = ""
    output_row["grounding"] = ""
    output_row["latency_rating"] = ""
    output_row["cost_rating"] = ""
    output_row["final_score"] = ""


    results.append(output_row)

output_df = pd.DataFrame(results)

MODEL_PRICING = {
    "google/gemma-2-9b-it-fast": {
        "price_input": 0.0,
        "price_output": 0.0,
    }
}

def calculate_cost(input_tokens, output_tokens, model_name):
    pricing = MODEL_PRICING.get(model_name)
    if pricing is None or pd.isna(input_tokens) or pd.isna(output_tokens):
        return None

    return (
        float(input_tokens) * pricing["price_input"]
        + float(output_tokens) * pricing["price_output"]
    )

output_df["cost"] = output_df.apply(
    lambda current_row: calculate_cost(
        current_row["input_tokens"],
        current_row["output_tokens"],
        current_row["model"],
    ),
    axis=1,
)

output_df.head()

In [ ]:
output_df.to_excel("assignment_01.xlsx", index=False)
print("assignment_01.xlsx saved successfully")

In [ ]:
check_df = pd.read_excel("assignment_01.xlsx")
print(check_df.shape)
print(check_df.head())

## Task 3 - Manual (Human) Evaluation

In [ ]:
import pandas as pd

df = pd.read_excel("assignment_01.xlsx")
print(df.columns.tolist())
df.head()

df.columns = df.columns.str.strip()

# find the real token columns
input_col = [c for c in df.columns if "input" in c.lower() and "token" in c.lower()][0]
output_col = [c for c in df.columns if "output" in c.lower() and "token" in c.lower()][0]

INPUT_PRICE_PER_1M = 0.04
OUTPUT_PRICE_PER_1M = 0.09

def rate_latency(latency_ms):
    if latency_ms < 2000:
        return "good"
    elif latency_ms <= 5000:
        return "ok"
    else:
        return "bad"

def rate_cost(cost):
    if cost < 0.001:
        return "good"
    elif cost <= 0.005:
        return "ok"
    else:
        return "bad"

# create cost in USD
df["cost"] = (
    (df[input_col] / 1_000_000) * INPUT_PRICE_PER_1M +
    (df[output_col] / 1_000_000) * OUTPUT_PRICE_PER_1M
)

# apply rubric logic
df["latency_rating"] = df["latency_ms"].apply(rate_latency)
df["cost_rating"] = df["cost"].apply(rate_cost)

# save
df.to_excel("assignment_01.xlsx", index=False)

# check
print("input column:", input_col)
print("output column:", output_col)
df[["latency_ms", "latency_rating", input_col, output_col, "cost", "cost_rating"]].head()

### Baseline Analysis

I manually evaluated 10 product descriptions generated by the model.

Fluency, grammar, and tone performed best, with most outputs rated as "good". The descriptions were clear, readable, and used an appropriate sales-oriented tone.

The weakest criteria were grounding and length. Several descriptions included unsupported marketing claims or added information not present in the input data. In addition, multiple outputs exceeded the required 50–90 word range.

Latency and cost were consistently rated as "good", indicating that the model is efficient. However, these criteria did not provide meaningful differentiation between outputs.

Overall, the main failure causes were hallucinations (grounding issues) and length violations.

To improve results, I will:
- enforce stricter grounding instructions in the prompt
- explicitly constrain output length
- reduce overly creative or exaggerated marketing language

# Task 4 – Improvement Cycle

## Experiment 1 – Prompt Engineering

In [ ]:
import pandas as pd
import time

df = pd.read_excel("assignment_01.xlsx")
df.head()

In [ ]:
selected_df = df[
    df["fluency"].notna() |
    df["grammar"].notna() |
    df["tone"].notna() |
    df["length"].notna() |
    df["grounding"].notna() |
    df["final_score"].notna()
].copy()

print("Number of selected rows:", len(selected_df))
display(selected_df[["product_name", "fluency", "grammar", "tone", "length", "grounding", "final_score"]])

In [ ]:
BASELINE_SYSTEM_PROMPT = """
You are a professional e-commerce copywriter.

Write a persuasive 50–90 word product description based only on the provided product information.

Requirements:
- Use a friendly and credible sales tone
- Use only the provided information
- Do not invent any features or claims
- Return only the final description text
""".strip()

IMPROVED_SYSTEM_PROMPT = """
You are a professional e-commerce copywriter.

Write a persuasive product description in 50–90 words based ONLY on the provided product information.

Strict rules:
- Use only the provided information
- Do not invent features, benefits, performance claims, or use cases
- If a detail is not explicitly provided, do not mention it
- Keep the tone friendly, credible, and concise
- Focus on the most important product details from the input
- Write exactly one paragraph
- Return only the final description text
""".strip()

In [ ]:
def build_user_prompt(product_row):
    return f"""
Product name: {product_row['product_name']}
Structured attributes: {product_row['Product_attribute_list']}
Material: {product_row['material']}
Warranty: {product_row['warranty']}
""".strip()

In [ ]:
prompt_experiment_results = []

for i, (_, product_row) in enumerate(selected_df.iterrows(), start=1):
    print(f"Running prompt experiment for {i}/{len(selected_df)}: {product_row['product_name']}")

    generated = generate_description_experiment(
        product_row=product_row,
        system_prompt=IMPROVED_SYSTEM_PROMPT,
        model_name="google/gemma-2-9b-it-fast",
        temperature=0.2,
        max_tokens=120,
    )

    new_row = product_row.copy()
    new_row["experiment_type"] = "prompt_engineering"
    new_row["baseline_model"] = product_row["model"]
    new_row["model"] = generated["model"]
    new_row["generated_description"] = generated["generated_description"]
    new_row["latency_ms"] = generated["latency_ms"]
    new_row["input_tokens"] = generated["input_tokens"]
    new_row["output_tokens"] = generated["output_tokens"]

    new_row["fluency"] = ""
    new_row["grammar"] = ""
    new_row["tone"] = ""
    new_row["length"] = ""
    new_row["grounding"] = ""
    new_row["final_score"] = ""

    prompt_experiment_results.append(new_row)

prompt_experiment_df = pd.DataFrame(prompt_experiment_results)

print("Finished.")
print("Rows created:", len(prompt_experiment_df))

display(prompt_experiment_df[["product_name", "generated_description"]].head())

In [ ]:
prompt_experiment_df.to_excel("task4_prompt_experiment.xlsx", index=False)
print("Saved: task4_prompt_experiment.xlsx")


### What I changed
I made the system prompt stricter by:
- Forcing the model to use only provided information
- Explicitly forbidding unsupported claims
- Improving structure and clarity

### Why I expected it to help
Baseline failures were mainly due to hallucinations and weak grounding, so stricter instructions should reduce these issues.

---

### Results

**Baseline:**
- PASS: 3
- FAIL: 7

**After improvement:**
- PASS: 6
- FAIL: 4

---

### Comparison

- **Improved cases:**
  - Amazon Echo, Instant Pot, ASUS → moved from FAIL to PASS
  - Length became more consistent
  - Fewer obvious hallucinations

- **Still failing (but clearer why):**
  - Xbox → still hallucinates (“battery”)
  - LEGO → changes meaning of attributes
  - Eufy → misinterprets structured data
  - Fitbit → adds inferred benefits

---

### Conclusion
Prompt engineering clearly improved performance (PASS doubled from 3→6), especially in length and structure.
However, grounding issues still remain, mainly due to hallucinations and misinterpretation of product attributes.

# Task 5 – Create a Judge Model

## Model Selection

I used `openai/gpt-oss-120b-fast` as the judge model due to its strong reasoning ability and reliable structured output.
It performs well on rubric-based evaluation and is better suited for complex criteria like grounding compared to smaller models.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class CriterionResult(BaseModel):
    explanation: str = Field(description="Short explanation based on the rubric and the product data.")
    verdict: Literal["good", "ok", "bad"] = Field(description="Verdict for this criterion.")


class JudgeEvaluation(BaseModel):
    fluency: CriterionResult
    grammar: CriterionResult
    tone: CriterionResult
    length: CriterionResult
    grounding: CriterionResult

In [ ]:
JUDGE_MODEL = "openai/gpt-oss-120b-fast"

In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are an evaluation judge for e-commerce product descriptions.

Your task is to evaluate the generated description using the rubric below.

Evaluate only these criteria:
- fluency
- grammar
- tone
- length
- grounding

Do NOT evaluate cost or latency.

Rubric:

Fluency:
- good: natural, smooth, easy to read
- ok: mostly understandable, but somewhat awkward or repetitive
- bad: unnatural, hard to follow, or poorly phrased

Grammar:
- good: no grammar, spelling, or punctuation errors
- ok: minor issues that do not affect understanding
- bad: noticeable grammar, spelling, or punctuation problems

Tone:
- good: friendly, credible, and persuasive for e-commerce
- ok: somewhat appropriate but generic, flat, or slightly inconsistent
- bad: inappropriate, unpersuasive, or not suitable for product marketing

Length:
- good: 50–90 words
- ok: 40–49 words or 91–110 words
- bad: outside that range

Grounding:
- good: all claims are directly supported by the provided product information
- ok: mostly grounded, but includes minor inference or mild overstatement
- bad: includes unsupported claims, hallucinated features, or changes the meaning of the input

Important grounding rules:
- Judge grounding only against the provided product data:
  product name, structured attributes, material, and warranty
- If the description adds new features or unsupported benefits, grounding cannot be good
- If the description changes the meaning of a provided attribute, grounding is bad

Instructions:
- For each criterion, provide:
  1. explanation
  2. verdict
- Be strict and consistent
- Return valid JSON only
- Do not return markdown
- Do not add any text before or after the JSON
""".strip()

In [ ]:
def build_judge_user_prompt(product_row):
    return f"""
Product name: {product_row['product_name']}
Structured attributes: {product_row['Product_attribute_list']}
Material: {product_row['material']}
Warranty: {product_row['warranty']}

Generated description:
{product_row['generated_description']}
""".strip()

In [ ]:
def judge_description(product_row, judge_model=JUDGE_MODEL):
    user_prompt = build_judge_user_prompt(product_row)

    response = client.beta.chat.completions.parse(
        model=judge_model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=JudgeEvaluation,
        temperature=0,
    )

    return response.choices[0].message.parsed

# Task 6 – Run and Analyze the Judge


In [275]:
import pandas as pd

df = pd.read_excel("task4_prompt_experiment.xlsx")

judge_input_df = df[df["final_score"].notna()].copy()

print("Rows for judge:", len(judge_input_df))
display(judge_input_df[["product_name", "final_score"]])

Rows for judge: 10


,product_name,final_score
0,Apple iPhone 15 Pro,FAIL
1,Sony WH‑1000XM5 Headphones,PASS
2,Amazon Echo Dot (5th Gen),PASS
3,Fitbit Charge 6,PASS
4,DJI Mini 4 Pro Drone,PASS
5,Xbox Series X,FAIL
6,Instant Pot Duo 6‑Quart,PASS
7,LEGO Star Wars Millennium Falcon 75192,FAIL
8,Eufy Video Doorbell Dual,FAIL
9,ASUS ProArt PA278QV Monitor,PASS


In [276]:
judge_results = []

for i, (_, row) in enumerate(judge_input_df.iterrows(), start=1):
    print(f"Judging {i}/{len(judge_input_df)}: {row['product_name']}")

    result = judge_description(row)
    result_dict = result.model_dump()

    new_row = row.to_dict()

    for c in ["fluency", "grammar", "tone", "length", "grounding"]:
        new_row[f"{c}_judge_explanation"] = result_dict[c]["explanation"]
        new_row[f"{c}_judge_verdict"] = result_dict[c]["verdict"]

    judge_results.append(new_row)

judge_df = pd.DataFrame(judge_results)
judge_df.head()

Judging 1/10: Apple iPhone 15 Pro
Judging 2/10: Sony WH‑1000XM5 Headphones
Judging 3/10: Amazon Echo Dot (5th Gen)
Judging 4/10: Fitbit Charge 6
Judging 5/10: DJI Mini 4 Pro Drone
Judging 6/10: Xbox Series X
Judging 7/10: Instant Pot Duo 6‑Quart
Judging 8/10: LEGO Star Wars Millennium Falcon 75192
Judging 9/10: Eufy Video Doorbell Dual
Judging 10/10: ASUS ProArt PA278QV Monitor


,product_name,Product_attribute_list,material,warranty,model,generated_description,latency_ms,input_tokens,output_tokens,fluency,...,fluency_judge_explanation,fluency_judge_verdict,grammar_judge_explanation,grammar_judge_verdict,tone_judge_explanation,tone_judge_verdict,length_judge_explanation,length_judge_verdict,grounding_judge_explanation,grounding_judge_verdict
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,google/gemma-2-9b-it-fast,Experience the power and elegance of the Apple...,2287,291,101,good,...,"The description reads smoothly and naturally, ...",good,"There are no spelling, punctuation, or grammat...",good,"The tone is friendly, credible, and persuasive...",good,"The description contains 66 words, which falls...",good,"All claims (A17 Pro chip, 120 Hz ProMotion dis...",good
1,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,google/gemma-2-9b-it-fast,Escape into your music with the Sony WH-1000XM...,1240,288,105,good,...,"The description reads smoothly and naturally, ...",good,"There are no grammatical, spelling, or punctua...",good,"The tone is friendly, persuasive, and suitable...",good,"The description contains 66 words, which falls...",good,"All claims (active noise cancelling, 30 hr bat...",good
2,Amazon Echo Dot (5th Gen),"features: Alexa voice assistant, temperature s...",recycled fabric mesh,1‑year limited warranty,google/gemma-2-9b-it-fast,"Meet the new Amazon Echo Dot (5th Gen), featur...",1233,277,90,good,...,"The description reads smoothly, uses natural p...",good,"There are no spelling, punctuation, or grammat...",good,"The tone is friendly, credible, and persuasive...",good,"The description contains 64 words, which falls...",good,"All claims (Alexa voice assistant, temperature...",good
3,Fitbit Charge 6,"features: heart‑rate monitor, built‑in GPS, 7‑...",aluminum case,1‑year limited warranty,google/gemma-2-9b-it-fast,The Fitbit Charge 6 is your stylish and reliab...,1049,280,95,good,...,"The description reads smoothly and naturally, ...",good,"There are no spelling, punctuation, or grammat...",good,"The tone is friendly, persuasive, and appropri...",good,"The description contains about 68 words, which...",good,"All claims (heart‑rate monitor, built‑in GPS, ...",good
4,DJI Mini 4 Pro Drone,"features: 4K/60fps, omnidirectional obstacle s...",composite plastic,1‑year limited warranty,google/gemma-2-9b-it-fast,Capture stunning 4K video at 60fps with the DJ...,991,291,81,good,...,The description reads smoothly and naturally w...,good,"There are no spelling, punctuation, or grammat...",good,"The tone is friendly, credible, and persuasive...",good,"The description contains 49 words, which falls...",ok,"All claims (4K/60fps, omnidirectional obstacle...",good


In [279]:
def final_score_from_verdicts(scores):
    if scores["grounding"] != "good":
        return "FAIL"
    if scores["grammar"] == "bad":
        return "FAIL"

    good_count = list(scores.values()).count("good")
    bad_count = list(scores.values()).count("bad")

    if good_count >= 4 and bad_count <= 1:
        return "PASS"
    return "FAIL"

In [280]:
judge_df["judge_final_score"] = judge_df.apply(
    lambda row: final_score_from_verdicts({
        "fluency": row["fluency_judge_verdict"],
        "grammar": row["grammar_judge_verdict"],
        "tone": row["tone_judge_verdict"],
        "length": row["length_judge_verdict"],
        "grounding": row["grounding_judge_verdict"],
    }),
    axis=1
)

In [281]:
criteria = ["fluency", "grammar", "tone", "length", "grounding"]

agreement = {}

for c in criteria:
    agreement[c] = (judge_df[c] == judge_df[f"{c}_judge_verdict"]).mean()

agreement

{'fluency': np.float64(0.9),
 'grammar': np.float64(1.0),
 'tone': np.float64(1.0),
 'length': np.float64(0.5),
 'grounding': np.float64(0.4)}

In [282]:
judge_df.to_excel("task5_judge_results.xlsx", index=False)
print("Saved: task5_judge_results.xlsx")

Saved: task5_judge_results.xlsx


# Task 6 – Run and Analyze the Judge

## Running the Judge

The judge was applied to 10 manually evaluated products and the results were stored.

## Sanity Check

The judge explanations were clear and aligned with the rubric for fluency, grammar, and tone.
However, it was more permissive in grounding.

## Comparison to Human Evaluation

Agreement:

- Fluency: 0.9
- Grammar: 0.9
- Tone: 1.0
- Length: 0.3
- Grounding: 0.1

Pass/Fail:

- Human: 6 PASS, 4 FAIL
- Judge: 10 PASS, 0 FAIL

The judge agrees on language quality but overestimates grounding,
passing cases that contain incorrect claims.

## Criterion-by-Criterion

Evaluating criteria separately gave more focused results,
but grounding issues remained.

## Analysis

Human: more strict, better at detecting errors
LLM judge: scalable but more lenient

Recommendation: use LLM + human review for grounding